In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

In [3]:
apenas_ano = []
for an in anos:
    ano = an.split("-")[0]
    apenas_ano.append(ano)
    # print(ano, type(ano))
    os.makedirs(f'score_piotroski/{ano}', exist_ok=True)
apenas_ano

['2025',
 '2024',
 '2023',
 '2022',
 '2021',
 '2020',
 '2019',
 '2018',
 '2017',
 '2016',
 '2015']

## Leitura dos dados base

In [4]:
basedados_ativos = Path('../../base_dados/brapi/retornos/retornos.csv')
basedados_ibov = Path('../../base_dados/retorno_ibov_2015_2026.csv') 

lista_piotroski = []

for filename in os.listdir(path='../../base_dados/brapi/piotroski/'):
    
    # 2. Reconstruct the full absolute or relative path to the file
    full_path = os.path.join('../../base_dados/brapi/piotroski/', filename)
    
    # 3. Check if the current item is actually a file (and not a subfolder)
    # if os.path.isfile(full_path):
        
    #     # 4. Open and process the file safely
    #     with open(full_path, "r", encoding="utf-8") as file:
    #         content = file.read()
    #         print(f"--- Content of {filename} ---")
    #         print(pd.read_csv(full_path))

    df = pd.read_csv(full_path).drop(columns=['Unnamed: 0']).set_index('endDate')
    dicio = {
        'ativo':filename,
        'data':df
    }

    lista_piotroski.append(dicio)

df_ativos=pd.read_csv(basedados_ativos).set_index(['date']).fillna(0)




In [5]:
lista_piotroski

[{'ativo': 'ABEV3',
  'data':             ABEV3
  endDate          
  2012-12-31      2
  2013-12-31      6
  2014-12-31      6
  2015-12-31      4
  2016-12-31      6
  2017-12-31      6
  2018-12-31      6
  2019-12-31      4
  2020-12-31      4
  2021-12-31      5
  2022-12-31      5
  2023-12-31      7
  2024-12-31      6
  2025-12-31      7},
 {'ativo': 'ALOS3',
  'data':             ALOS3
  endDate          
  2012-12-31      2
  2013-12-31      4
  2014-12-31      5
  2015-12-31      5
  2016-12-31      6
  2017-12-31      5
  2018-12-31      7
  2019-12-31      6
  2020-12-31      6
  2021-12-31      7
  2022-12-31      6
  2023-12-31      5
  2024-12-31      7
  2025-12-31      8},
 {'ativo': 'ANIM3',
  'data':             ANIM3
  endDate          
  2012-12-31      3
  2013-12-31      6
  2014-12-31      7
  2015-12-31      2
  2016-12-31      4
  2017-12-31      9
  2018-12-31      4
  2019-12-31      3
  2020-12-31      4
  2021-12-31      5
  2022-12-31      8
  2023-12-31

## Iteração para verificação se o ativo da lista piotroski tem retorno naquele ano para entrar no score do ano

In [6]:
dict_df = {}
for i in range(len(apenas_ano)):
    # print(apenas_ano[i])
    df = df_ativos[df_ativos.index.str.startswith(apenas_ano[i])]
    df = df.drop(columns=df.columns[(df == 0).all()])

    print(apenas_ano[i],"---- Quantidade de ativos",len(df.columns))
    dict_df[apenas_ano[i]] = df



2025 ---- Quantidade de ativos 78
2024 ---- Quantidade de ativos 78
2023 ---- Quantidade de ativos 78
2022 ---- Quantidade de ativos 77
2021 ---- Quantidade de ativos 77
2020 ---- Quantidade de ativos 73
2019 ---- Quantidade de ativos 72
2018 ---- Quantidade de ativos 70
2017 ---- Quantidade de ativos 70
2016 ---- Quantidade de ativos 64
2015 ---- Quantidade de ativos 62


In [7]:
SCORE_MAX = 9

for ano in apenas_ano:
    ativos_do_ano = dict_df[ano].columns          # já filtrados, sem os zerados

    linha = {}
    for j in lista_piotroski:
        ativo = j['ativo'].replace('.csv', '')
        if ativo not in ativos_do_ano:
            continue

        df_ativo = j['data']
        col = next((c for c in (j['ativo'], ativo) if c in df_ativo.columns),
                   df_ativo.columns[-1])

        alvo = [d for d in df_ativo.index if str(d).startswith(ano)]
        if alvo:
            linha[ativo] = df_ativo.loc[alvo[0], col] / SCORE_MAX

    df = pd.DataFrame([linha], index=[ano])
    minimo = df.min(axis=1)
    maximo = df.max(axis=1)
    df_ano = df.sub(minimo, axis=0).div(maximo - minimo, axis=0)

    df_ano.to_csv(f'score_piotroski/{ano}/piotroski_{ano}.csv', index_label='ano')

    print(f'{ano}: {len(ativos_do_ano)} negociando, {len(linha)} com score')

2025: 78 negociando, 76 com score
2024: 78 negociando, 76 com score
2023: 78 negociando, 76 com score
2022: 77 negociando, 75 com score
2021: 77 negociando, 75 com score
2020: 73 negociando, 72 com score
2019: 72 negociando, 71 com score
2018: 70 negociando, 69 com score
2017: 70 negociando, 69 com score
2016: 64 negociando, 64 com score
2015: 62 negociando, 62 com score
